# Exploring Emotional Profiles in 1.38M TMDB Movies

**Dataset:** [TMDB Movie VAD + Emotion Intensity Scores](https://www.kaggle.com/datasets/bdelanghe/tmdb-movie-vad-emotion-scores)
**Source:** [TMDB Movies Dataset](https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies) enriched with NRC lexicons by Saif M. Mohammad (NRC Canada)

Each movie is scored on 12 affective dimensions derived from plot keywords:
- **VAD** — valence (positive/negative), arousal (calm/excited), dominance (submissive/powerful)
- **8 emotions** — anger, anticipation, disgust, fear, joy, sadness, surprise, trust (0–1 intensity)

Scores represent *keyword associations*, not audience reaction. ~23% of movies have keyword coverage.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Resolve input path — works whether notebook is created from dataset page or pushed via CLI
from pathlib import Path
matches = sorted(Path('/kaggle/input').rglob('movie_vad_scores.csv'))
INPUT_PATH = matches[0]
print(f'Input: {INPUT_PATH}')

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f'Rows: {len(df):,}   Columns: {len(df.columns)}')

## 1. Schema and coverage

In [ ]:
EMOTIONS = ['anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']
VAD = ['valence', 'arousal', 'dominance']

print('Sentiment distribution:')
print(df['sentiment'].value_counts().to_string())
print(f'\nMovies with keyword coverage: {(df.sentiment != "unknown").sum():,} ({(df.sentiment != "unknown").mean():.1%})')
print('\nEmotion score means (scored movies only):')
scored = df[df['sentiment'] != 'unknown']
print(scored[EMOTIONS].mean().round(3).to_string())
print('\nVAD means:')
print(scored[VAD].mean().round(3).to_string())

## 2. Top films by emotion

In [ ]:
# Popular movies only (vote_count >= 1000) with keyword coverage (>= 5 keywords)
has_kw = df[
    df['keywords'].notna() &
    (df['keywords'].str.count(',') >= 4) &
    (df['vote_count'] >= 500)
]
print(f'Movies in pool: {len(has_kw):,}')

print('\nMost joyful (by keyword associations):')
display(has_kw.nlargest(8, 'joy')[['title', 'genres', 'release_date', 'joy', 'sadness', 'sentiment']].reset_index(drop=True))

print('\nMost fear-associated:')
display(has_kw.nlargest(8, 'fear')[['title', 'genres', 'release_date', 'fear', 'anger', 'sentiment']].reset_index(drop=True))

print('\nMost negative valence:')
display(has_kw.nsmallest(8, 'valence')[['title', 'genres', 'release_date', 'valence', 'arousal', 'sentiment']].reset_index(drop=True))

## 3. Genre emotional fingerprints

In [ ]:
genre_df = (
    df[df['genres'].notna() & (df['valence'].notna())]
    .assign(genre=df['genres'].str.split(', '))
    .explode('genre')
)

FEATURES = VAD + EMOTIONS
genre_means = genre_df.groupby('genre')[FEATURES].mean()
genre_means['n_movies'] = genre_df.groupby('genre').size()
genre_means = genre_means[genre_means['n_movies'] >= 50].copy()

display(genre_means[VAD + ['n_movies']].sort_values('valence', ascending=False).round(3).head(12))

## 4. Valence × Arousal map by genre

Each point is a genre. Position = emotional tone (valence × arousal). Size = dominance.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

top_genres = genre_means.sort_values('n_movies', ascending=False).head(18)
colors = cm.tab20(np.linspace(0, 1, len(top_genres)))

for (genre, row), color in zip(top_genres.iterrows(), colors):
    size = 200 + row['dominance'] * 600
    ax.scatter(row['valence'], row['arousal'], s=size, color=color, alpha=0.85, zorder=3)
    ax.annotate(genre, (row['valence'], row['arousal']),
                fontsize=9, ha='center', va='bottom',
                xytext=(0, 7), textcoords='offset points')

ax.axvline(0.5, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.axhline(0.5, color='gray', linewidth=0.8, linestyle='--', alpha=0.5)
ax.set_xlabel('Valence  (0 = negative  →  1 = positive)', fontsize=11)
ax.set_ylabel('Arousal  (0 = calm  →  1 = excited)', fontsize=11)
ax.set_title('Genre Emotional Map — NRC VAD scores from TMDB keywords\n(dot size = dominance)', fontsize=12)
ax.set_xlim(0.3, 0.75); ax.set_ylim(0.3, 0.65)
ax.text(0.32, 0.31, 'Low energy, negative', fontsize=8, color='gray', alpha=0.7)
ax.text(0.60, 0.62, 'High energy, positive', fontsize=8, color='gray', alpha=0.7)
plt.tight_layout()
plt.show()

## Case study: The Sixth Sense & Alien

Why did these films score the way they did?

In [ ]:
sixth = df[df['title'].str.contains('Sixth Sense', case=False, na=False)].iloc[0]

print('Title:   ', sixth['title'])
print('Keywords:', sixth['keywords'])
print()
print('--- VAD scores ---')
for d in ['valence', 'arousal', 'dominance']:
    print(f'  {d:12s}: {sixth[d]:.3f}')
print()
print('--- Emotion scores ---')
for e in EMOTIONS:
    print(f'  {e:14s}: {sixth[e]:.3f}')
print()
print('--- Keyword groups (VAD valence buckets) ---')
kw_cols = ['positive_keywords', 'negative_keywords', 'unknown_keywords']
missing = [c for c in kw_cols if c not in df.columns]
if missing:
    print(f'  NOTE: columns not in this dataset version: {missing}')
else:
    print('  positive:', sixth['positive_keywords'])
    print('  negative:', sixth['negative_keywords'])
    print('  unknown: ', sixth['unknown_keywords'])
print()
print('Sentiment:', sixth['sentiment'])

# ── Alien ─────────────────────────────────────────────────────────────────
alien = df[df['title'].str.lower() == 'alien'].iloc[0]

print()
print('Title:   ', alien['title'])
print('Keywords:', alien['keywords'])
print()
print('--- VAD scores ---')
for d in ['valence', 'arousal', 'dominance']:
    print(f'  {d:12s}: {alien[d]:.3f}')
print()
print('--- Emotion scores ---')
for e in EMOTIONS:
    print(f'  {e:14s}: {alien[e]:.3f}')
print()
print('--- Keyword groups (VAD valence buckets) ---')
kw_cols = ['positive_keywords', 'negative_keywords', 'unknown_keywords']
missing = [c for c in kw_cols if c not in df.columns]
if missing:
    print(f'  NOTE: columns not in this dataset version: {missing}')
else:
    print('  positive:', alien['positive_keywords'])
    print('  negative:', alien['negative_keywords'])
    print('  unknown: ', alien['unknown_keywords'])
print()
print('Sentiment:', alien['sentiment'])

## 5. Emotions over time

How have movie emotional profiles shifted by decade?

In [ ]:
decade_df = (
    df[df['release_date'].notna() & df['fear'].notna()]
    .assign(decade=lambda d: (pd.to_datetime(d['release_date'], errors='coerce').dt.year // 10 * 10).astype('Int64'))
    .query('decade >= 1950 and decade <= 2020')
)

decade_means = decade_df.groupby('decade')[EMOTIONS + VAD].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: emotion intensity over time
for emo in EMOTIONS:
    axes[0].plot(decade_means['decade'], decade_means[emo], marker='o', label=emo)
axes[0].set_title('Emotion intensity by decade')
axes[0].set_xlabel('Decade')
axes[0].set_ylabel('Mean intensity (0–1)')
axes[0].legend(fontsize=8, ncol=2)
axes[0].set_xticks(decade_means['decade'])

# Right: VAD over time
for dim in VAD:
    axes[1].plot(decade_means['decade'], decade_means[dim], marker='s', label=dim)
axes[1].axhline(0, color='gray', lw=0.7, linestyle='--')
axes[1].set_title('VAD dimensions by decade')
axes[1].set_xlabel('Decade')
axes[1].set_ylabel('Mean score (bipolar −1 to +1)')
axes[1].legend()
axes[1].set_xticks(decade_means['decade'])

plt.tight_layout()
plt.show()
print(decade_means[['decade'] + EMOTIONS + VAD].to_string(index=False))


## 6. Revenue × valence

Do positive or negative movies gross more?

In [ ]:
rev_df = df[
    (df['revenue'] > 1_000_000) &
    df['valence'].notna()
].copy()
rev_df['log_revenue'] = np.log10(rev_df['revenue'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter
sc = axes[0].scatter(rev_df['valence'], rev_df['log_revenue'],
                     alpha=0.15, s=6, c=rev_df['valence'], cmap='RdYlGn', vmin=-1, vmax=1)
axes[0].set_xlabel('Valence (−1 to +1)')
axes[0].set_ylabel('log₁₀(Revenue USD)')
axes[0].set_title('Valence vs Box-office Revenue')
axes[0].axvline(0, color='gray', lw=0.7, linestyle='--')
plt.colorbar(sc, ax=axes[0], label='valence')

# Box by sentiment label
sentiment_order = ['positive', 'neutral', 'negative']
groups = [rev_df[rev_df['sentiment'] == s]['log_revenue'] for s in sentiment_order]
axes[1].boxplot(groups, labels=sentiment_order, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_title('Revenue by sentiment label')
axes[1].set_ylabel('log₁₀(Revenue USD)')

plt.tight_layout()
plt.show()

print("Median revenue by sentiment:")
print(rev_df.groupby('sentiment')['revenue'].median().apply(lambda x: f'${x:,.0f}').to_string())


## 7. Rating × emotion

Does trust predict high ratings? Does disgust predict low ones?

In [ ]:
rated_df = df[df['vote_count'] >= 100].dropna(subset=['vote_average'] + EMOTIONS)

corr = rated_df[EMOTIONS + ['vote_average']].corr()['vote_average'].drop('vote_average').sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart of correlations
colors = ['#d73027' if v < 0 else '#1a9850' for v in corr.values]
axes[0].barh(corr.index, corr.values, color=colors)
axes[0].axvline(0, color='black', lw=0.8)
axes[0].set_title('Pearson r: emotion vs vote_average')
axes[0].set_xlabel('Correlation')

# Scatter for strongest +/− emotion
best = corr.idxmax()
worst = corr.idxmin()
for ax, emo, color in [(axes[1], best, '#1a9850')]:
    ax.scatter(rated_df[emo], rated_df['vote_average'], alpha=0.1, s=5, c=color)
    ax.set_xlabel(f'{emo} intensity')
    ax.set_ylabel('vote_average')
    ax.set_title(f'Rating vs {emo} (r={corr[emo]:.3f})')

plt.tight_layout()
plt.show()
print(f"Strongest positive predictor: {best} (r={corr[best]:.3f})")
print(f"Strongest negative predictor: {worst} (r={corr[worst]:.3f})")


## 8. Dominance by genre

High dominance = power/control (action, superhero). Low dominance = vulnerability (horror, drama).

In [ ]:
dom_genre = (
    df[df['genres'].notna() & df['dominance'].notna()]
    .assign(genre=df['genres'].str.split(', '))
    .explode('genre')
    .groupby('genre')
    .agg(dominance=('dominance','mean'), n=('dominance','count'))
    .query('n >= 200')
    .sort_values('dominance')
)

fig, ax = plt.subplots(figsize=(9, 7))
colors = cm.RdYlGn((dom_genre['dominance'] - dom_genre['dominance'].min()) /
                   (dom_genre['dominance'].max() - dom_genre['dominance'].min()))
bars = ax.barh(dom_genre.index, dom_genre['dominance'], color=colors)
ax.axvline(0, color='gray', lw=0.8, linestyle='--')
ax.set_title('Mean dominance by genre (bipolar −1 to +1)')
ax.set_xlabel('Dominance score')
plt.tight_layout()
plt.show()
print(dom_genre[['dominance','n']].to_string())


## 9. Bi-emotion scatter

Films high on *both* joy **and** fear simultaneously — horror-comedy territory. And anger + trust (war films).

In [ ]:
popular = df[(df['vote_count'] >= 500) & df['joy'].notna() & df['fear'].notna() & df['genres'].notna()].copy()

# Pick the first listed genre for colouring
popular['primary_genre'] = popular['genres'].str.split(', ').str[0]
top_genres_list = popular['primary_genre'].value_counts().head(10).index
pop_top = popular[popular['primary_genre'].isin(top_genres_list)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

palette = cm.tab10(np.linspace(0, 1, len(top_genres_list)))
genre_color = dict(zip(top_genres_list, palette))

for genre in top_genres_list:
    g = pop_top[pop_top['primary_genre'] == genre]
    axes[0].scatter(g['joy'], g['fear'], alpha=0.35, s=10,
                    color=genre_color[genre], label=genre)
axes[0].set_xlabel('Joy intensity')
axes[0].set_ylabel('Fear intensity')
axes[0].set_title('Joy vs Fear (colored by genre)')
axes[0].legend(fontsize=7, markerscale=2)

for genre in top_genres_list:
    g = pop_top[pop_top['primary_genre'] == genre]
    axes[1].scatter(g['anger'], g['trust'], alpha=0.35, s=10,
                    color=genre_color[genre], label=genre)
axes[1].set_xlabel('Anger intensity')
axes[1].set_ylabel('Trust intensity')
axes[1].set_title('Anger vs Trust (colored by genre)')
axes[1].legend(fontsize=7, markerscale=2)

plt.tight_layout()
plt.show()


## 10. Emotion radar by genre

All 8 NRC emotions per genre — spider chart.

In [ ]:
from matplotlib.patches import FancyArrowPatch

radar_genres = ['Horror', 'Romance', 'Comedy', 'Action', 'Drama', 'Animation']
genre_emo = (
    df[df['genres'].notna() & df['fear'].notna()]
    .assign(genre=df['genres'].str.split(', '))
    .explode('genre')
    .groupby('genre')[EMOTIONS]
    .mean()
)
radar_data = genre_emo.loc[[g for g in radar_genres if g in genre_emo.index]]

angles = np.linspace(0, 2 * np.pi, len(EMOTIONS), endpoint=False).tolist()
angles += angles[:1]  # close

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
colors = cm.tab10(np.linspace(0, 1, len(radar_data)))

for (genre, row), color in zip(radar_data.iterrows(), colors):
    values = row[EMOTIONS].tolist() + [row[EMOTIONS[0]]]
    ax.plot(angles, values, 'o-', linewidth=1.5, label=genre, color=color)
    ax.fill(angles, values, alpha=0.08, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(EMOTIONS, size=11)
ax.set_title('Emotion profiles by genre', size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()


## 11. Most neutral genre

Which genre has the lowest total emotion intensity across all 8 dimensions?

In [ ]:
genre_intensity = (
    df[df['genres'].notna() & df['fear'].notna()]
    .assign(genre=df['genres'].str.split(', '))
    .explode('genre')
    .groupby('genre')
    .agg(n=('fear', 'count'), **{e: (e, 'mean') for e in EMOTIONS})
    .query('n >= 200')
)
# Sum all emotion means into one score
genre_intensity['total_intensity'] = genre_intensity[EMOTIONS].sum(axis=1)
genre_intensity = genre_intensity.sort_values('total_intensity')

fig, ax = plt.subplots(figsize=(9, 7))
colors_bar = cm.RdYlGn_r(
    (genre_intensity['total_intensity'] - genre_intensity['total_intensity'].min()) /
    (genre_intensity['total_intensity'].max() - genre_intensity['total_intensity'].min())
)
ax.barh(genre_intensity.index, genre_intensity['total_intensity'], color=colors_bar)
ax.set_title('Total emotion intensity by genre (sum of 8 NRC emotions)')
ax.set_xlabel('Sum of mean emotion scores')
plt.tight_layout()
plt.show()
print("\nMost neutral (lowest intensity):")
print(genre_intensity[['total_intensity','n']].head(5).to_string())
print("\nMost emotionally intense:")
print(genre_intensity[['total_intensity','n']].tail(5).to_string())


## 12. Keywords driving extreme scores

Which keywords appear most in the top 1% of movies by each emotion?

In [ ]:
from collections import Counter

# Base rates across all movies with keywords
_base_counts = Counter()
for _kws in df['keywords'].dropna().str.lower().str.split(', '):
    _base_counts.update(_kws)
_total_movies = sum(_base_counts.values())

def top_keywords_for(emotion, n=20, percentile=99, min_count=50):
    """Return keywords with highest lift in top-percentile movies for this emotion.
    Lift = (freq in extreme movies) / (base rate across all movies).
    Filters out keywords appearing fewer than min_count times overall.
    """
    threshold = df[emotion].quantile(percentile / 100)
    extreme = df[df[emotion] >= threshold].dropna(subset=['keywords'])
    n_extreme = len(extreme)

    emo_counts = Counter()
    for kws in extreme['keywords'].str.lower().str.split(', '):
        emo_counts.update(kws)

    lift = {}
    for kw, count in emo_counts.items():
        if _base_counts[kw] < min_count:
            continue
        p_emo = count / n_extreme
        p_all = _base_counts[kw] / _total_movies
        lift[kw] = p_emo / p_all

    return pd.Series(lift).sort_values(ascending=False).head(n)

focus_emotions = ['fear', 'joy', 'sadness', 'anger', 'trust', 'anticipation']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, emo in zip(axes.flat, focus_emotions):
    kw_lift = top_keywords_for(emo)
    ax.barh(kw_lift.index[::-1], kw_lift.values[::-1], color='steelblue')
    ax.set_title(f'Top keywords — {emo} (top 1%, by lift)')
    ax.set_xlabel('Lift (vs base rate)')
    ax.tick_params(labelsize=8)

plt.suptitle('Keywords most over-represented in extreme emotion scores (lift over base rate)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## Caveats

1. **Scores are relative, not absolute.** Valence 0.65 means "more positive-associated than 0.55" — not "65% positive."
2. **Keyword associations, not audience emotion.** A horror film with "fear" keywords scores high on fear — that's an association, not a review.
3. **~77% of movies have no keywords** and are scored `unknown`. Analyses above use the 23% with coverage.
4. **Bipolar composition is lossy for unknown phrases.** When a keyword isn't in the MWE list, it's tokenized and each token's bipolar score is averaged. This can wash out signal — "not happy" averages a negative and a positive rather than composing correctly. See: Mohammad et al., [Sentiment Composition of Words with Opposing Polarities](https://www.saifmohammad.com/WebPages/SCL.html) (SCL) for why token-level averaging on a bipolar scale is unreliable for phrases not in the lexicon.

See also: Mohammad (2020), [Practical and Ethical Considerations in the Effective use of Emotion and Sentiment Lexicons](https://www.saifmohammad.com/WebDocs/EmoLex-Ethics-Data-Statement.pdf).